In [ ]:
import requests
import pandas as pd
import time

# 1. 인증 및 헤더 설정
headers = {
    'accept': 'application/json, text/plain, */*',
    'accept-language': 'ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7',
    'origin': 'https://www.yogiyo.co.kr',
    'referer': 'https://www.yogiyo.co.kr/',
    'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/148.0.0.0 Safari/537.36',
    'x-api-key': 'qua9EeW1ohth4ain',
    'x-ygy-app-version': '9.0.0',
    'x-ygy-os-type': 'IOS',
    'x-ygy-route': 'v2',
}

# 2. 파라미터 초기 설정
params = {
    'adm_code': '1162069500',
    'customer_id': '923892007',
    'lat': '37.4869016',
    'lng': '126.92737994',
    'length': '60',
    'sort': 'RANK_DESC',
    'vertical_types': 'FOOD',
    'membership_code': 'NONE',
    'serving_types': 'VD',
    'use_bargainyo': 'false'
}

url = 'https://api.yogiyo.co.kr/shopyo/v1/shops'

all_shops = []
start_val = 0
is_end = False

print("데이터 수집 시작")

# 3. 데이터가 없을 때까지 반복 수행
while not is_end:
    params['start'] = str(start_val)
    
    try:
        response = requests.get(url, params=params, headers=headers)
        
        if response.status_code == 200:
            data = response.json()
            shops_list = data.get('shops', [])
            
            # 더 이상 가져올 데이터가 없는 경우 루프 종료
            if not shops_list:
                print(f"\n수집 완료: {start_val} 지점에서 데이터가 더 이상 없습니다.")
                is_end = True
                break
                
            all_shops.extend(shops_list)
            print(f"현재 위치: {start_val} / 누적 상점 수: {len(all_shops)}", end='\r')
            
            # 다음 페이지를 위해 인덱스 증가
            start_val += 60
            
        elif response.status_code in [401, 403]:
            print("\n인증 키 만료 혹은 접근 차단이 발생했습니다.")
            break
        else:
            print(f"\n오류 발생 (상태 코드: {response.status_code})")
            break
            
    except Exception as e:
        print(f"\n네트워크 예외 발생: {e}")
        break
        
    # 서버 부하 방지를 위한 대기 시간
    time.sleep(1.5)

# 4. 결과 저장
if all_shops:
    df = pd.DataFrame(all_shops)
    print("\n수집 결과 요약")
    print(f"총 상점 수: {len(df)}")
    
    # 분석용 주요 컬럼 존재 확인 및 출력
    #cols = ['name', 'review_avg', 'delivery_fee', 'min_order_amount']
    #available = [c for c in cols if c in df.columns]
    #print(df[available].head())
    
    # 파일 저장
    df.to_csv("yogiyo_full_data.csv", index=False, encoding='utf-8-sig')
    print("파일 저장 완료: yogiyo_full_data.csv")
else:
    print("\n수집된 데이터가 없습니다.")

데이터 수집 시작
현재 위치: 1860 / 누적 상점 수: 1895
수집 완료: 1920 지점에서 데이터가 더 이상 없습니다.

수집 결과 요약
총 상점 수: 1895
               name
0       대치동엄마도시락-본점
1  앵그리포테이토치킨&버거-신림점
2          호호솥밥-관악점
3          육회한날연어어때
4           괴짜반점-본점
파일 저장 완료: yogiyo_full_data.csv


In [1]:
import pandas as pd
df = pd.read_csv('yogiyo_full_data.csv')
df



,id,name,location,vertical_type,vertical_sub_type,vendor_categories,tags,bpr_ranking,image,review,open_status,franchise,serving_type,distance,catalog_vendor_id,exposure_type,ad_id,point,representative_menus
0,1502665,대치동엄마도시락-본점,"{'lat': '37.4839342', 'lng': '126.93936786'}",FOOD,NaN,"['찜/탕', '한식', '도시락/죽', '포장']",['RELAYO'],NaN,"{'logo_url': None, 'thumbnail_url': 'https://r...","{'average_rating': 4.9, 'count': 195, 'image_c...","{'current_open_status': 'OPENING_TIME', 'next_...",NaN,"{'vd': {'ypx_minimum_order_amount': 0, 'estima...",1108.02,NaN,CONTRACT,NaN,{'total': 0},"[{'id': 2038071069, 'name': '[BEST] 연탄 벌집(삼겹) ..."
1,1505082,앵그리포테이토치킨&버거-신림점,"{'lat': '37.4837321', 'lng': '126.92774951'}",FOOD,NaN,"['피자/양식', '버거', '분식', '포장', '1인분주문']",[],NaN,{'logo_url': 'https://rev-static.yogiyo.co.kr/...,"{'average_rating': 5.0, 'count': 203, 'image_c...","{'current_open_status': 'OPENING_TIME', 'next_...",NaN,"{'vd': {'ypx_minimum_order_amount': None, 'est...",353.94,NaN,CONTRACT,NaN,{'total': 0},"[{'id': 2098832456, 'name': '나홀로 1인 세트(시그니처)',..."
2,1517961,호호솥밥-관악점,"{'lat': '37.4854819', 'lng': '126.93790844'}",FOOD,NaN,"['한식', '고기/구이', '일식/돈까스', '포장', '프랜차이즈', '신규맛집']",['NEW'],NaN,{'logo_url': 'https://rev-static.yogiyo.co.kr/...,"{'average_rating': 5.0, 'count': 20, 'image_co...","{'current_open_status': 'OPENING_TIME', 'next_...",NaN,"{'vd': {'ypx_minimum_order_amount': None, 'est...",942.28,NaN,CONTRACT,NaN,{'total': 0},"[{'id': 2152991873, 'name': '[시그니처] 스테이크 솥밥', ..."
3,568358,육회한날연어어때,"{'lat': '37.48203242479008', 'lng': '126.92966...",FOOD,NaN,"['포장', '일식/돈까스', '야식', '회/초밥']",[],NaN,"{'logo_url': None, 'thumbnail_url': 'https://r...","{'average_rating': 5.0, 'count': 15593, 'image...","{'current_open_status': 'OPENING_TIME', 'next_...",NaN,"{'vd': {'ypx_minimum_order_amount': 0, 'estima...",581.24,NaN,CONTRACT,NaN,{'total': 0},"[{'id': 930778160, 'name': '스테이크육회초밥', 'image_..."
4,1214991,괴짜반점-본점,"{'lat': '37.4839734', 'lng': '126.93930939'}",FOOD,NaN,"['한식', '중국집', '야식', '찜/탕', '포장']",['RELAYO'],NaN,{'logo_url': 'https://rev-static.yogiyo.co.kr/...,"{'average_rating': 4.8, 'count': 3847, 'image_...","{'current_open_status': 'OPENING_TIME', 'next_...",NaN,"{'vd': {'ypx_minimum_order_amount': 0, 'estima...",1101.79,NaN,CONTRACT,NaN,{'total': 0},"[{'id': 935692199, 'name': '[라드로 볶은] 찐한 풍미 짜장면..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1890,1335521,헬로크랩-개봉역점,"{'lat': '37.4929162', 'lng': '126.85993106'}",FOOD,NaN,"['찜/탕', '한식', '회/초밥', '포장', '프랜차이즈']",['CESCO'],NaN,{'logo_url': 'https://rev-static.yogiyo.co.kr/...,"{'average_rating': 5.0, 'count': 8, 'image_cou...","{'current_open_status': 'TEMP_HOLIDAY', 'next_...",NaN,"{'vd': {'ypx_minimum_order_amount': 0, 'estima...",5988.40,NaN,CONTRACT,NaN,{'total': 0},"[{'id': 991304079, 'name': '[서울, 경기최저가]A급 활 대게..."
1891,1287151,니드쿠키(kneadcookie),"{'lat': '37.5028137', 'lng': '126.89775928'}",FOOD,NaN,"['분식', '카페/디저트']",[],NaN,"{'logo_url': None, 'thumbnail_url': 'https://r...","{'average_rating': 5.0, 'count': 3, 'image_cou...","{'current_open_status': 'CLOSED', 'next_open_s...",NaN,"{'vd': {'ypx_minimum_order_amount': None, 'est...",3155.87,NaN,CONTRACT,NaN,{'total': 0},"[{'id': 937760707, 'name': '아메리카노', 'image_pat..."
1892,1086464,초장집-영등포점,"{'lat': '37.517175509084986', 'lng': '126.9075...",FOOD,NaN,"['찜/탕', '한식', '회/초밥', '야식', '포장', '프랜차이즈']",['CESCO'],NaN,"{'logo_url': None, 'thumbnail_url': 'https://r...","{'average_rating': 4.9, 'count': 57, 'image_co...","{'current_open_status': 'TEMP_HOLIDAY', 'next_...",NaN,"{'vd': {'ypx_minimum_order_amount': None, 'est...",3792.88,NaN,CONTRACT,NaN,{'total': 0},"[{'id': 933603786, 'name': '연포탕', 'image_path'..."
1893,1227416,깡우동-시흥사거리점,"{'lat': '37.4520775', 'lng': '126.90266415'}",FOOD,NaN,"['찜/탕', '한식', '분식', '야식', '포장', '프랜차이즈']",[],NaN,"{'logo_url': None, 'thumbnail_url': 'https://r...","{'average_rating': 4.5, 'count': 2, 'image_cou...","{'current_open_status': 'TEMP_HOLIDAY', 'next_...",N

In [2]:
df.columns

Index(['id', 'name', 'location', 'vertical_type', 'vertical_sub_type',
       'vendor_categories', 'tags', 'bpr_ranking', 'image', 'review',
       'open_status', 'franchise', 'serving_type', 'distance',
       'catalog_vendor_id', 'exposure_type', 'ad_id', 'point',
       'representative_menus'],
      dtype='object')

In [3]:
cols = ["name","location","vendor_categories","review"]

In [4]:
new_df = df[cols]

In [5]:
import ast

# 1. 문자열을 실제 딕셔너리 객체로 변환
new_df['location'] = new_df['location'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
new_df['review'] = new_df['review'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

# 2. 딕셔너리 안의 키들을 새로운 컬럼으로 확장
review_cols = new_df['review'].apply(pd.Series)
location_cols = new_df['location'].apply(pd.Series)
# 3. 기존 데이터프레임과 합치기 (기존 'review' 컬럼은 삭제 가능)
new_df = pd.concat([new_df, review_cols,location_cols], axis=1)

C:\Users\seon\AppData\Local\Temp\ipykernel_9236\2677093267.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['location'] = new_df['location'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
C:\Users\seon\AppData\Local\Temp\ipykernel_9236\2677093267.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['review'] = new_df['review'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)


In [6]:
new_df.loc[new_df["name"]=="육회한날연어어때",:]

,name,location,vendor_categories,review,average_rating,count,image_count,reply_count,lat,lng
3,육회한날연어어때,"{'lat': '37.48203242479008', 'lng': '126.92966...","['포장', '일식/돈까스', '야식', '회/초밥']","{'average_rating': 5.0, 'count': 15593, 'image...",5.0,15593.0,14295.0,13506.0,37.48203242479008,126.92966049495293


In [7]:
df['vendor_categories'][0]

"['찜/탕', '한식', '도시락/죽', '포장']"

In [8]:
new_df.to_csv("yogiyo__data.csv", index=False, encoding='utf-8-sig')

In [ ]:
import requests
import pandas as pd
import time
from tqdm import tqdm

# 1. API 키 설정
KAKAO_API_KEY = 'f62a560324bef030e4a4ef2a1ba39808'

def get_full_address(lat, lng):
    url = "https://dapi.kakao.com/v2/local/geo/coord2address.json"
    headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
    params = {"x": float(lng), "y": float(lat)}
    
    try:
        response = requests.get(url, headers=headers, params=params)
        if response.status_code == 200:
            data = response.json()
            if data['documents']:
                addr_info = data['documents'][0]
                
                # 도로명 주소가 있으면 사용, 없으면 지번 주소 사용
                road = addr_info.get('road_address')
                lot = addr_info.get('address')
                
                full_addr = road['address_name'] if road else lot['address_name']
                
                return {
                    'full_address': full_addr,
                    'sido': lot['region_1depth_name'],
                    'gu': lot['region_2depth_name'],
                    'dong': lot['region_3depth_name']
                }
    except:
        pass
    return {'full_address': None, 'sido': None, 'gu': None, 'dong': None}

# 2. 데이터 불러오기
df = pd.read_csv('yogiyo__data.csv')

# 3. 전체 데이터 변환 (tqdm으로 진행 상황 확인)
results = []
print("주소 변환 작업을 시작합니다...")
for index, row in tqdm(df.iterrows(), total=len(df), desc="변환 진행 중"):
    addr_data = get_full_address(row['lat'], row['lng'])
    results.append(addr_data)
    # API 안정성을 위해 0.05초 대기 (초당 약 20건 처리)
    time.sleep(0.05)

# 4. 데이터 결합 및 저장
addr_df = pd.DataFrame(results)
final_df = pd.concat([df.reset_index(drop=True), addr_df], axis=1)

# 서울 데이터만 따로 추출하여 저장 (태블로 시각화용)
seoul_df = final_df[final_df['sido'] == '서울']
seoul_df.to_csv('yogiyo_final_seoul.csv', index=False, encoding='utf-8-sig')

print(f"\n작업 완료! 저장된 서울 데이터 수: {len(seoul_df)}건")

주소 변환 작업을 시작합니다...


변환 진행 중:   0%|          | 0/1895 [00:00<?, ?it/s]

변환 진행 중: 100%|██████████| 1895/1895 [03:46<00:00,  8.35it/s]


작업 완료! 저장된 서울 데이터 수: 0건


In [28]:
# 서울 데이터만 따로 추출하여 저장 (태블로 시각화용)
seoul_df = final_df[final_df['sido'] == '서울']
seoul_df.to_csv('yogiyo_final_seoul.csv', index=False, encoding='utf-8-sig')

print(f"\n작업 완료! 저장된 서울 데이터 수: {len(seoul_df)}건")


작업 완료! 저장된 서울 데이터 수: 1894건


In [29]:
seoul_df

,name,location,vendor_categories,review,average_rating,count,image_count,reply_count,lat,lng,full_address,sido,gu,dong
0,대치동엄마도시락-본점,"{'lat': '37.4839342', 'lng': '126.93936786'}","['찜/탕', '한식', '도시락/죽', '포장']","{'average_rating': 4.9, 'count': 195, 'image_c...",4.9,195.0,157.0,179.0,37.483934,126.939368,서울특별시 관악구 봉천로 356,서울,관악구,봉천동
1,앵그리포테이토치킨&버거-신림점,"{'lat': '37.4837321', 'lng': '126.92774951'}","['피자/양식', '버거', '분식', '포장', '1인분주문']","{'average_rating': 5.0, 'count': 203, 'image_c...",5.0,203.0,205.0,43.0,37.483732,126.927750,서울특별시 관악구 남부순환로 1594,서울,관악구,신림동
2,호호솥밥-관악점,"{'lat': '37.4854819', 'lng': '126.93790844'}","['한식', '고기/구이', '일식/돈까스', '포장', '프랜차이즈', '신규맛집']","{'average_rating': 5.0, 'count': 20, 'image_co...",5.0,20.0,20.0,0.0,37.485482,126.937908,서울특별시 관악구 봉천로25길 4,서울,관악구,봉천동
3,육회한날연어어때,"{'lat': '37.48203242479008', 'lng': '126.92966...","['포장', '일식/돈까스', '야식', '회/초밥']","{'average_rating': 5.0, 'count': 15593, 'image...",5.0,15593.0,14295.0,13506.0,37.482032,126.929660,서울특별시 관악구 신림로 309,서울,관악구,신림동
4,괴짜반점-본점,"{'lat': '37.4839734', 'lng': '126.93930939'}","['한식', '중국집', '야식', '찜/탕', '포장']","{'average_rating': 4.8, 'count': 3847, 'image_...",4.8,3847.0,3532.0,2868.0,37.483973,126.939309,서울특별시 관악구 봉천로 356,서울,관악구,봉천동
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1890,헬로크랩-개봉역점,"{'lat': '37.4929162', 'lng': '126.85993106'}","['찜/탕', '한식', '회/초밥', '포장', '프랜차이즈']","{'average_rating': 5.0, 'count': 8, 'image_cou...",5.0,8.0,10.0,0.0,37.492916,126.859931,서울특별시 구로구 개봉로20길 101,서울,구로구,개봉동
1891,니드쿠키(kneadcookie),"{'lat': '37.5028137', 'lng': '126.89775928'}","['분식', '카페/디저트']","{'average_rating': 5.0, 'count': 3, 'image_cou...",5.0,3.0,4.0,0.0,37.502814,126.897759,서울특별시 영등포구 대림로44길 11,서울,영등포구,대림동
1892,초장집-영등포점,"{'lat': '37.517175509084986', 'lng': '126.9075...","['찜/탕', '한식', '회/초밥', '야식', '포장', '프랜차이즈']","{'average_rating': 4.9, 'count': 57, 'image_co...",4.9,57.0,55.0,45.0,37.517176,126.907570,서울특별시 영등포구 영중로4길 10-1,서울,영등포구,영등포동3가
1893,깡우동-시흥사거리점,"{'lat': '37.4520775', 'lng': '126.90266415'}","['찜/탕', '한식', '분식', '야식', '포장', '프랜차이즈']","{'average_rating': 4.5, 'count': 2, 'image_cou...",4.5,2.0,1.0,0.0,37.452078,126.902664,서울특별시 금천구 시흥대로52길 16,서울,금천구,시흥동


In [33]:
len(seoul_df["vendor_categories"].unique())

1032